In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import pantab as pt

In [2]:
# Variables to update:

# cleaned claims data files
state_folder = "Virginia"
payer_folder = "Cigna Analysis"

# rate files
rates_file = "All_Rates_2008_2025"
asp_file = "asp_drug_pricing_2024_Q1_2026_Q2"
us_file = "US_2009_2026"

In [3]:
# Pull in the cleaned billing data from Tableau Prep Builder output
base_folder = Path(
    r"C:\Users\pirat\Dropbox\Consulting Inc\Advanced Dermatology\1. Payor Data & Contracts"
)

payer_name = payer_folder.replace(" Analysis", "")
base_name = f"{state_folder}_{payer_name}_Billing_Cleaned"

hyper_file = base_folder / state_folder / payer_folder / "Tableau" / "Hyper" / f"{base_name}.hyper"

if not hyper_file.exists():
    raise FileNotFoundError(f"Hyper file not found: {hyper_file}")

billing_clean = pt.frame_from_hyper(hyper_file, table="Extract")


In [4]:
# Rates Data (use rates_file path)
rates_path = Path(
    r"C:\Users\pirat\Dropbox\Consulting Inc\Studies and Articles\CMS RVU and Conversion Factors\PAF\Export"
) 

# Pull in the rates
paf_rates = pd.read_parquet(
    rates_path / f"{rates_file}.parquet",
    engine="pyarrow"
    # ,filters=filters
).drop(columns=["Year","State", "PCTC Indicator", "F Amount", "Multiple Surgery Indicator"])



In [5]:
# import the US Rates and concat to all_rates
us_rates = pd.read_parquet(
    rates_path / f"{us_file}.parquet",
    engine="pyarrow"
).drop(columns=["Year","State", "Description"])

all_rates = pd.concat([paf_rates, us_rates], ignore_index=True)

In [6]:
# pull in asp data
asp_path = Path(
    r"C:\Users\pirat\Dropbox\Consulting Inc\Studies and Articles\CMS RVU and Conversion Factors\ASP\Export"
) / f"{asp_file}.parquet"

asp_rates = pd.read_parquet(
    asp_path,
    engine="pyarrow"
)

In [7]:
# build match list from billing codes
match_codes = (
    billing_clean["CPT_Code_Full"]
    .astype("string")
    .str.strip()
    .dropna()
    .drop_duplicates()
)
# normalize rate-side code columns
all_rates["Code_Full"] = all_rates["Code_Full"].astype("string").str.strip()
asp_rates["hcpcs_code"] = asp_rates["hcpcs_code"].astype("string").str.strip()

# keep only matching rows
all_rates = all_rates.loc[
    all_rates["Code_Full"].isin(match_codes)
].copy()

all_asp = asp_rates.loc[
    asp_rates["hcpcs_code"].isin(match_codes)
].copy()

In [8]:
# normalize fields
billing_clean["CPT_Code_Full"] = billing_clean["CPT_Code_Full"].astype("string").str.strip()
billing_clean["Date"] = pd.to_datetime(billing_clean["Date"].astype("string"), errors="coerce")

all_asp["hcpcs_code"] = all_asp["hcpcs_code"].astype("string").str.strip()
all_asp["year_quarter"] = all_asp["year_quarter"].astype("string").str.strip()
all_asp["payment_limit"] = pd.to_numeric(all_asp["payment_limit"], errors="coerce")

# build billing quarter key
billing_clean["_year_quarter"] = (
    billing_clean["Date"].dt.year.astype("Int64").astype("string")
    + "_Q" +
    billing_clean["Date"].dt.quarter.astype("Int64").astype("string")
)

# slim ASP lookup
asp_lookup = (
    all_asp[["hcpcs_code", "year_quarter", "payment_limit"]]
    .drop_duplicates(["hcpcs_code", "year_quarter"])
    .rename(columns={
        "hcpcs_code": "CPT_Code_Full",
        "year_quarter": "_year_quarter",
        "payment_limit": "Benchmark_ASP",
    })
)

# merge in Benchmark_ASP
billing_clean = billing_clean.merge(
    asp_lookup,
    how="left",
    on=["CPT_Code_Full", "_year_quarter"]
)

# optional cleanup
billing_clean = billing_clean.drop(columns=["_year_quarter"])

In [9]:
# target_cols = [
#     "Year_FS",
#     "MAC",
#     "Locality",
#     "State_FS",
#     "Locality_Name",
#     "HCPCS Code",
#     "Modifier",
#     "Code_Full",
#     "NF Amount",
#     "Status Code",
#     "year_quarter",
#     "Start_Date",
#     "End_Date",
#     "Source",
# ]

# # base PAF table
# all_rates = all_rates.copy()
# all_rates["Source"] = "PAF"

# # ASP table aligned to PAF structure
# asp_to_add = all_asp.rename(
#     columns={
#         "hcpcs_code": "HCPCS Code",
#         "payment_limit": "NF Amount",
#     }
# ).copy()

# asp_to_add["Source"] = "ASP"


# # make sure both tables have all target columns
# for col in target_cols:
#     if col not in all_rates.columns:
#         all_rates[col] = pd.NaT if col in ["Start_Date", "End_Date"] else pd.NA
#     if col not in asp_to_add.columns:
#         asp_to_add[col] = pd.NaT if col in ["Start_Date", "End_Date"] else pd.NA

# # concat back into all_rates
# all_rates = pd.concat(
#     [all_rates[target_cols], asp_to_add[target_cols]],
#     ignore_index=True
# )

In [17]:
# add State to beginning of locality for easier filtering
all_rates["Locality_Name"] = all_rates["State_FS"] + " - " + all_rates["Locality_Name"]

In [18]:
# output file path
output_dir = base_folder / state_folder / payer_folder / "Tableau"
xlsx_base = output_dir / "Excel Backups" / f"{state_folder}_{payer_name}_All_Rates"
hyper_base = output_dir / "Hyper" / f"{state_folder}_{payer_name}_All_Rates"

exports = {
    "xlsx":  xlsx_base.with_suffix(".xlsx"),
    "hyper": hyper_base.with_suffix(".hyper"),
}

for p in exports.values():
    p.parent.mkdir(parents=True, exist_ok=True)


# one simple schema for Hyper-safe typing
schema = {
    "Year_FS": "string",
    "MAC": "string",
    "Locality": "string",
    "State_FS": "string",
    "Locality_Name": "string",
    "HCPCS Code": "string",
    "Modifier": "string",
    "Code_Full": "string",
    "NF Amount": "float64",
    "Status Code": "string"
}

all_rates_hyper = all_rates.copy().reset_index(drop=True)

for col, dtype in schema.items():
    if col not in all_rates_hyper.columns:
        continue

    if dtype == "string":
        all_rates_hyper[col] = all_rates_hyper[col].astype("string").str.strip()
    elif dtype == "float64":
        all_rates_hyper[col] = pd.to_numeric(all_rates_hyper[col], errors="coerce")
    elif dtype == "datetime64[ns]":
        all_rates_hyper[col] = pd.to_datetime(all_rates_hyper[col], errors="coerce")


# optional: handle any bools generically
bool_cols = all_rates_hyper.select_dtypes(include=["bool", "boolean"]).columns
if len(bool_cols):
    all_rates_hyper[bool_cols] = all_rates_hyper[bool_cols].astype("Int8")


writers = {
    "xlsx":  lambda df, p: df.to_excel(p, index=False),
    "hyper": lambda df, p: pt.frame_to_hyper(df, str(p), table="all_rates"),
}

for fmt, path in exports.items():
    df_to_write = all_rates_hyper if fmt == "hyper" else all_rates
    writers[fmt](df_to_write, path)

In [16]:
# Make a Hyper-safe copy (mainly: booleans)

import pantab


xlsx_base = output_dir / "Excel Backups" / f"{state_folder}_{payer_name}_Billing_Cleaned"
hyper_base = output_dir / "Hyper" / f"{state_folder}_{payer_name}_Billing_Cleaned"

exports = {
    "xlsx":  xlsx_base.with_suffix(".xlsx"),
    "hyper": hyper_base.with_suffix(".hyper"),
}

billing_hyper = billing_clean.copy().reset_index(drop=True)

bool_cols = billing_hyper.select_dtypes(include=["bool", "boolean"]).columns
if len(bool_cols):
    billing_hyper[bool_cols] = billing_hyper[bool_cols].astype("Int8")

writers = {
    "xlsx":  lambda df, p: df.to_excel(p, index=False),
    "hyper": lambda df, p: pantab.frame_to_hyper(df, str(p), table="Extract"),
}

for fmt, path in exports.items():
    path.parent.mkdir(parents=True, exist_ok=True)
    df_to_write = billing_hyper if fmt == "hyper" else billing_clean
    writers[fmt](df_to_write, path)